In [1]:
import numpy as np
import pandas as pd

# Fixed seed so the whole notebook is reproducible
rng = np.random.default_rng(12345)

def sh_analysis(n):
    """SH Analysis: Exponential, mean 10 days"""
    return rng.exponential(scale=10, size=n)

def data_extract(n):
    """Data Extract: Normal, mean 15 days, sd 3 days"""
    return rng.normal(loc=15, scale=3, size=n)

def dev_exploration(n):
    """Dev Exploration: Normal, mean 3 days, sd 0.5 days"""
    return rng.normal(loc=3, scale=0.5, size=n)

def dev_modeling(n):
    """Dev Modeling: Poisson, mean 30 days"""
    return rng.poisson(lam=30, size=n)

def process_integration(n):
    """Process Integration: Normal, mean 50 days, sd 20 days"""
    return rng.normal(loc=50, scale=20, size=n)

def data_pipeline(n):
    """Data Pipeline: Poisson, mean 100 days"""
    return rng.poisson(lam=100, size=n)

def model_productionization(n):
    """Model Productionization: Normal, mean 30 days, sd 5 days"""
    return rng.normal(loc=30, scale=5, size=n)


## Part (b): Run 1,000,000 simulations of the full process

**Accounting for dependencies:** the 7 steps run sequentially (each step only starts once
the previous one finishes), so the total time for a single simulated run of the whole
process is just the sum of that run's 7 step durations:

$$T_{total} = T_{SH} + T_{DE} + T_{DX} + T_{DM} + T_{PI} + T_{DP} + T_{MP}$$

To respect this, we draw **one full set of 7 values per trial** (by calling each step's
function once with `n = 1,000,000`, which produces one array per step, all aligned by
trial/row index), then sum the seven arrays *element-wise*. Trial `i`'s total is the sum
of element `i` from every step's array — i.e. every trial's total is built from values
that all belong to that same trial, which is exactly what "accounting for the
dependencies" requires. We do **not** simulate each step independently across unrelated
sets of a million trials and try to recombine summary stats afterward — that would break
the trial-by-trial pairing we need for the contribution analysis in part (b)(ii).

In [2]:
n = 1_000_000

steps = {
    "SH Analysis":              sh_analysis(n),
    "Data Extract":              data_extract(n),
    "Dev Exploration":            dev_exploration(n),
    "Dev Modeling":               dev_modeling(n),
    "Process Integration":        process_integration(n),
    "Data Pipeline":              data_pipeline(n),
    "Model Productionization":    model_productionization(n),
}

steps_df = pd.DataFrame(steps)          # one row per trial, one column per step
total_time = steps_df.sum(axis=1)       # element-wise sum -> total time per trial

steps_df.head()

,SH Analysis,Data Extract,Dev Exploration,Dev Modeling,Process Integration,Data Pipeline,Model Productionization
0,1.841326,20.541162,3.122724,28,63.969584,100,31.397022
1,6.450271,15.543402,2.900656,21,40.853165,90,29.299870
2,46.902187,18.245066,3.963850,33,65.276676,88,25.240769
3,4.185587,13.683493,2.340527,34,23.119024,108,33.160371
4,5.110474,17.673741,2.461912,23,23.511896,112,30.669453


### b.ii.1 — Average, median, and 95% CI for the overall process time



In [3]:
mean_t   = total_time.mean()
median_t = total_time.median()
sd_t     = total_time.std(ddof=1)

# 1) 95% probability interval (2.5th/97.5th percentile of simulated outcomes)
pi_lo, pi_hi = np.percentile(total_time, [2.5, 97.5])

# 2) 95% CI on the estimated mean (standard error of the mean)
se = sd_t / np.sqrt(n)
ci_lo, ci_hi = mean_t - 1.96 * se, mean_t + 1.96 * se

print(f"Mean total process time    : {mean_t:.2f} days")
print(f"Median total process time  : {median_t:.2f} days")
print(f"Std dev of total time      : {sd_t:.2f} days")
print()
print(f"95% probability interval (spread of outcomes) : [{pi_lo:.2f}, {pi_hi:.2f}] days")
print(f"95% CI on the mean estimate (precision of avg): [{ci_lo:.3f}, {ci_hi:.3f}] days")

Mean total process time    : 238.02 days
Median total process time  : 237.60 days
Std dev of total time      : 25.80 days

95% probability interval (spread of outcomes) : [188.66, 290.00] days
95% CI on the mean estimate (precision of avg): [237.974, 238.075] days


### b.ii — Which step contributes most to overall time, and to uncertainty?



In [4]:
contrib = pd.DataFrame({
    "mean_days": steps_df.mean(),
    "var_days2": steps_df.var(ddof=1),
})
contrib["pct_of_total_mean"] = 100 * contrib["mean_days"] / contrib["mean_days"].sum()
contrib["pct_of_total_var"]  = 100 * contrib["var_days2"] / contrib["var_days2"].sum()

contrib_sorted_by_time = contrib.sort_values("pct_of_total_mean", ascending=False)
contrib_sorted_by_time.round(2)

,mean_days,var_days2,pct_of_total_mean,pct_of_total_var
Data Pipeline,100.00,100.02,42.01,15.03
Process Integration,50.03,400.73,21.02,60.24
Dev Modeling,30.00,30.05,12.60,4.52
Model Productionization,30.00,25.05,12.60,3.77
Data Extract,15.00,9.00,6.30,1.35
SH Analysis,10.01,100.16,4.20,15.06
Dev Exploration,3.00,0.25,1.26,0.04


In [5]:
top_time_step = contrib["pct_of_total_mean"].idxmax()
top_uncertainty_step = contrib["pct_of_total_var"].idxmax()

print(f"Step contributing MOST to overall time        : {top_time_step} "
      f"({contrib.loc[top_time_step, 'pct_of_total_mean']:.1f}% of total mean)")
print(f"Step contributing MOST to overall uncertainty  : {top_uncertainty_step} "
      f"({contrib.loc[top_uncertainty_step, 'pct_of_total_var']:.1f}% of total variance)")

Step contributing MOST to overall time        : Data Pipeline (42.0% of total mean)
Step contributing MOST to overall uncertainty  : Process Integration (60.2% of total variance)
